In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("healthcare") \
    .config("spark.sql.warehouse.dir", "hdfs:///user/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

26/05/05 17:36:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [19]:
spark.conf.set("spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") 
spark.conf.set("spark.hadoop.fs.gs.auth.service.account.enable", "true") 

In [20]:
df = spark.read \
    .option("header", "true") \
    .csv("gs://ak_sparkbucket/source/")

df.show()

+----------+---+------+--------------+---------------------+--------------+
|patient_id|age|gender|diagnosis_code|diagnosis_description|diagnosis_date|
+----------+---+------+--------------+---------------------+--------------+
|      P301| 34|     M|          D123|               Cancer|    2026-05-04|
|      P302| 69|     M|          C345|               Cancer|    2026-05-04|
|      P303| 48|     F|          H234|             Diabetes|    2026-05-04|
|      P304| 46|     F|          C345|             Diabetes|    2026-05-04|
|      P305| 31|     M|          D123|  High Blood Pressure|    2026-05-04|
|      P306| 34|     M|          C345|  High Blood Pressure|    2026-05-04|
|      P307| 60|     M|          H234|             Diabetes|    2026-05-04|
|      P308| 59|     F|          H234|               Cancer|    2026-05-04|
|      P309| 33|     F|          C345|  High Blood Pressure|    2026-05-04|
|      P310| 62|     M|          H234|               Cancer|    2026-05-04|
|      P311|

In [21]:
# Check for null values in each column
null_counts = df.agg(
    *[sum(col(column).isNull().cast("int")).alias(f"{column}_null_count") for column in df.columns]
)

# Check for data types
data_type_checks = [col(column).cast("string").alias(f"{column}_type_check") for column in df.columns]

# Apply the data type checks
df_check = df.select(data_type_checks)

# Show the results of the checks
print("Null Counts:")
null_counts.show()

print("Data Type Checks:")
df_check.show()

Null Counts:


+---------------------+--------------+-----------------+-------------------------+--------------------------------+-------------------------+
|patient_id_null_count|age_null_count|gender_null_count|diagnosis_code_null_count|diagnosis_description_null_count|diagnosis_date_null_count|
+---------------------+--------------+-----------------+-------------------------+--------------------------------+-------------------------+
|                    0|             0|                0|                        0|                               0|                        0|
+---------------------+--------------+-----------------+-------------------------+--------------------------------+-------------------------+

Data Type Checks:


+---------------------+--------------+-----------------+-------------------------+--------------------------------+-------------------------+
|patient_id_type_check|age_type_check|gender_type_check|diagnosis_code_type_check|diagnosis_description_type_check|diagnosis_date_type_check|
+---------------------+--------------+-----------------+-------------------------+--------------------------------+-------------------------+
|                 P301|            34|                M|                     D123|                          Cancer|               2026-05-04|
|                 P302|            69|                M|                     C345|                          Cancer|               2026-05-04|
|                 P303|            48|                F|                     H234|                        Diabetes|               2026-05-04|
|                 P304|            46|                F|                     C345|                        Diabetes|               2026-05-04|
|     

In [22]:
# Group by diagnosis_code and gender, and calculate the count for each group
gender_counts = df.groupBy("diagnosis_code","diagnosis_description","gender").agg(count("patient_id").alias("counter"))

# Pivot the data to get Male and Female counts as separate columns
gender_pivoted = gender_counts.groupBy("diagnosis_code","diagnosis_description").pivot("gender").agg(
    coalesce(sum(when(col("gender") == "M", col("counter"))), lit(0)).alias("Males"),
    coalesce(sum(when(col("gender") == "F", col("counter"))), lit(0)).alias("Females"))

# Drop F_Males and M_females columns from gender_pivoted DataFrame
gender_pivoted = gender_pivoted.drop("F_Males","M_Females")

# Calculate the gender ratio
gender_ratio = gender_pivoted.withColumn("Gender_Ratio", col("M_Males") / col("F_Females"))

# Show updated DataFrame
gender_ratio.show()

+--------------+---------------------+---------+-------+------------------+
|diagnosis_code|diagnosis_description|F_Females|M_Males|      Gender_Ratio|
+--------------+---------------------+---------+-------+------------------+
|          H234|             Diabetes|       28|     29|1.0357142857142858|
|          H234|  High Blood Pressure|       27|     33|1.2222222222222223|
|          C345|  High Blood Pressure|       27|     21|0.7777777777777778|
|          C345|             Diabetes|       22|     29|1.3181818181818181|
|          H234|               Cancer|       23|     32| 1.391304347826087|
|          D123|             Diabetes|       26|     27|1.0384615384615385|
|          D123|               Cancer|       26|     32|1.2307692307692308|
|          D123|  High Blood Pressure|       28|     31|1.1071428571428572|
|          C345|               Cancer|       30|     29|0.9666666666666667|
+--------------+---------------------+---------+-------+------------------+



In [28]:
stage_path = "hdfs:///tmp/output/stage/"
target_path = "hdfs:///tmp/output/target/"

# 4. Create STAGE Table (Hive External Table)

spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS healthcare.stage_disease_ratio (
    diagnosis_code STRING,
    diagnosis_description STRING,
    F_Females BIGINT,
    M_Males BIGINT,
    Gender_Ratio DOUBLE
)
STORED AS PARQUET
LOCATION '{stage_path}'
""")


# 6. Write to STAGE
gender_ratio.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(stage_path)

spark.sql("DROP TABLE IF EXISTS healthcare.target_disease_ratio")

# 7. Create TARGET Table
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS healthcare.target_disease_ratio (
    diagnosis_code STRING,
    diagnosis_description STRING,
    F_Females BIGINT,
    M_Males BIGINT,
    Gender_Ratio DOUBLE
)
STORED AS PARQUET
LOCATION '{target_path}'
""")


# 8. UPSERT LOGIC (Simulated)
# Load existing target data (if exists)
try:
    target_df = spark.read.parquet(target_path)
except:
    target_df = spark.createDataFrame([], gender_ratio.schema)

# Merge (Upsert using key: diagnosis_code)
final_df = target_df.union(gender_ratio) \
    .dropDuplicates(["diagnosis_code"])


# 9. Write back to TARGET (Overwrite = Upsert effect)
final_df.write \
    .mode("overwrite") \
    .parquet(target_path)

spark.catalog.clearCache()
spark.sql("REFRESH TABLE healthcare.target_disease_ratio")

# 11. Show Results
spark.sql("SELECT * FROM healthcare.target_disease_ratio").show()


+--------------+---------------------+---------+-------+------------------+
|diagnosis_code|diagnosis_description|F_Females|M_Males|      Gender_Ratio|
+--------------+---------------------+---------+-------+------------------+
|          C345|  High Blood Pressure|       27|     21|1.2857142857142858|
|          D123|             Diabetes|       26|     27|0.9629629629629629|
|          H234|             Diabetes|       28|     29|0.9655172413793104|
+--------------+---------------------+---------+-------+------------------+



In [29]:
df.createOrReplaceTempView("patient_data")

# Use SQL to find the top 3 common diseases
top3_query = """
    SELECT
        ROW_NUMBER() OVER (ORDER BY count(*) DESC) AS Rank,
        diagnosis_code,
        diagnosis_description
    FROM
        patient_data
    GROUP BY
        diagnosis_code, diagnosis_description
    ORDER BY
        Rank
    LIMIT 3
"""

top3 = spark.sql(top3_query)

# Show the resulting DataFrame
top3.show()

26/05/05 18:01:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:01:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:01:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:01:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:01:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:01:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:01:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+--------------+---------------------+
|Rank|diagnosis_code|diagnosis_description|
+----+--------------+---------------------+
|   1|          H234|  High Blood Pressure|
|   2|          D123|  High Blood Pressure|
|   3|          C345|               Cancer|
+----+--------------+---------------------+



In [34]:
stage_path = "hdfs:///tmp/output/stage/"
target_path = "hdfs:///tmp/output/target/"

# 4. Create STAGE Table (Hive External Table)

spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS healthcare.stage_top3 (
    Rank INT,
    Diagnosis_code STRING,
    Diagnosis_description STRING
)
STORED AS PARQUET
LOCATION '{stage_path}'
""")


# 6. Write to STAGE
top3.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(stage_path)

spark.sql("DROP TABLE IF EXISTS healthcare.target_top3")

# 7. Create TARGET Table
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS healthcare.target_top3 (
    Rank INT,
    Diagnosis_code STRING,
    Diagnosis_description STRING
)
STORED AS PARQUET
LOCATION '{target_path}'
""")


# 8. UPSERT LOGIC (Simulated)
# Load existing target data (if exists)
try:
    target_df = spark.read.parquet(target_path)
except:
    target_df = spark.createDataFrame([], top3.schema)



# Merge (Upsert using key: diagnosis_code)
final_df = target_df.unionByName(top3, allowMissingColumns=True) \
    .dropDuplicates(["Diagnosis_code"])


# 9. Write back to TARGET (Overwrite = Upsert effect)
final_df.write \
    .mode("overwrite") \
    .parquet(target_path)

spark.catalog.clearCache()
spark.sql("REFRESH TABLE healthcare.target_top3")

# 11. Show Results
spark.sql("SELECT * FROM healthcare.target_top3").show()

26/05/05 18:22:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:22:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:22:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:22:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:22:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:22:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:22:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:22:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:22:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:22:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:22:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:22:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:22:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+--------------+---------------------+
|Rank|Diagnosis_code|Diagnosis_description|
+----+--------------+---------------------+
|NULL|          C345|  High Blood Pressure|
|NULL|          D123|             Diabetes|
|NULL|          H234|             Diabetes|
+----+--------------+---------------------+



In [35]:
# Use SQL to create age buckets
distro_query = """
    SELECT
        diagnosis_code,
        diagnosis_description,
        CASE
            WHEN age >= 30 AND age < 40 THEN '30-40'
            WHEN age >= 40 AND age < 50 THEN '41-50'
            WHEN age >= 50 AND age < 60 THEN '51-60'
            WHEN age >= 60 THEN '61+'
        END AS age_category,
        COUNT(patient_id) AS patient_count,
        CONCAT(diagnosis_code, '_', 
            CASE
               WHEN age >= 30 AND age < 40 THEN '30-40'
               WHEN age >= 40 AND age < 50 THEN '41-50'
               WHEN age >= 50 AND age < 60 THEN '51-60'
               WHEN age >= 60 THEN '61+'
            END) AS code_age_category
    FROM
        patient_data
    GROUP BY
        diagnosis_code, diagnosis_description, age_category
"""

distro = spark.sql(distro_query)

# Show the resulting DataFrame
distro.show()

+--------------+---------------------+------------+-------------+-----------------+
|diagnosis_code|diagnosis_description|age_category|patient_count|code_age_category|
+--------------+---------------------+------------+-------------+-----------------+
|          D123|  High Blood Pressure|       30-40|           14|       D123_30-40|
|          D123|             Diabetes|         61+|           21|         D123_61+|
|          H234|  High Blood Pressure|         61+|           18|         H234_61+|
|          C345|               Cancer|       30-40|           15|       C345_30-40|
|          D123|               Cancer|       30-40|           14|       D123_30-40|
|          C345|             Diabetes|       30-40|           14|       C345_30-40|
|          H234|               Cancer|       41-50|            9|       H234_41-50|
|          C345|               Cancer|       51-60|           14|       C345_51-60|
|          H234|               Cancer|       30-40|           16|       H234

In [39]:
stage_path = "hdfs:///tmp/output/stage/"
target_path = "hdfs:///tmp/output/target/"

# 4. Create STAGE Table (Hive External Table)

spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS healthcare.stage_age_distro (
    Diagnosis_code STRING,
    Diagnosis_description STRING,
    age_category STRING,
    patient_count INT,
    code_age_category STRING
)
STORED AS PARQUET
LOCATION '{stage_path}'
""")


# 6. Write to STAGE
top3.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(stage_path)

spark.sql("DROP TABLE IF EXISTS healthcare.target_age_distro")

# 7. Create TARGET Table
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS healthcare.target_age_distro (
    Diagnosis_code STRING,
    Diagnosis_description STRING,
    age_category STRING,
    patient_count INT,
    code_age_category STRING
)
STORED AS PARQUET
LOCATION '{target_path}'
""")


# 8. UPSERT LOGIC (Simulated)
# Load existing target data (if exists)
try:
    target_df = spark.read.parquet(target_path)
except:
    target_df = spark.createDataFrame([], distro.schema)



# Merge (Upsert using key: diagnosis_code)
final_df = target_df.unionByName(distro, allowMissingColumns=True) \
    .dropDuplicates(["Diagnosis_code"])


# 9. Write back to TARGET (Overwrite = Upsert effect)
final_df.write \
    .mode("overwrite") \
    .parquet(target_path)

spark.catalog.clearCache()
spark.sql("REFRESH TABLE healthcare.target_age_distro")

# 11. Show Results
spark.sql("SELECT * FROM healthcare.target_age_distro").show()

26/05/05 18:34:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:34:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/05 18:34:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:34:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:34:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:34:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:34:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/05/05 18:34:44 WARN TaskSetManager: Lost task 0.0 in stage 238.0 (TID 217) (spark-clusterak-w-1.europe-central2-a.c.project-aedd3f2d-4596-4437-9dd.internal executor 15): org.apache.spark.SparkException: Parquet column cannot be converted in file hdfs://spark-clusterak-m/tmp/output/target/part-00000-bd5aebb4-7bbc-4671-aaf9-9ff8e49f827f-c000.snappy.parquet. Column: [patient_count], Expected: int, Found: INT64.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unsupportedSchemaColumnConvertError(QueryExecutionErrors.scala:855)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:287)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:759)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql

Py4JJavaError: An error occurred while calling o1234.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 238.0 failed 4 times, most recent failure: Lost task 0.3 in stage 238.0 (TID 220) (spark-clusterak-w-1.europe-central2-a.c.project-aedd3f2d-4596-4437-9dd.internal executor 15): org.apache.spark.SparkException: Parquet column cannot be converted in file hdfs://spark-clusterak-m/tmp/output/target/part-00000-bd5aebb4-7bbc-4671-aaf9-9ff8e49f827f-c000.snappy.parquet. Column: [patient_count], Expected: int, Found: INT64.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unsupportedSchemaColumnConvertError(QueryExecutionErrors.scala:855)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:287)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:759)
	at org.

In [40]:
# Use SQL to flag senior citizens
senior_query = """
    SELECT
        patient_id,age,Case when age>=60 THEN 'Y'
        ELSE 'N' END as senior_citizen_flag
    FROM
        patient_data
"""

senior = spark.sql(senior_query)

# Show the resulting DataFrame
senior.show()

+----------+---+-------------------+
|patient_id|age|senior_citizen_flag|
+----------+---+-------------------+
|      P301| 34|                  N|
|      P302| 69|                  Y|
|      P303| 48|                  N|
|      P304| 46|                  N|
|      P305| 31|                  N|
|      P306| 34|                  N|
|      P307| 60|                  Y|
|      P308| 59|                  N|
|      P309| 33|                  N|
|      P310| 62|                  Y|
|      P311| 69|                  Y|
|      P312| 67|                  Y|
|      P313| 65|                  Y|
|      P314| 43|                  N|
|      P315| 66|                  Y|
|      P316| 67|                  Y|
|      P317| 64|                  Y|
|      P318| 41|                  N|
|      P319| 52|                  N|
|      P320| 48|                  N|
+----------+---+-------------------+
only showing top 20 rows



In [42]:
from google.cloud import storage

bucket_name = "ak_sparkbucket"
source_prefix = "source/"
archive_prefix = "archieve/"  # (you may want to fix spelling: "archive")

client = storage.Client()
bucket = client.bucket(bucket_name)

# List all files in source folder
blobs = bucket.list_blobs(prefix=source_prefix)

for blob in blobs:
    if blob.name.endswith(".csv"):
        print(f"{blob.name} moved to archive folder")

        # destination path
        new_name = blob.name.replace("source/", "archieve/")

        # copy to new location
        new_blob = bucket.copy_blob(blob, bucket, new_name)

        # delete original
        blob.delete()

source/health_data_20260501.csv moved to archive folder
source/health_data_20260502.csv moved to archive folder
source/health_data_20260503.csv moved to archive folder
source/health_data_20260504.csv moved to archive folder
source/health_data_20260505.csv moved to archive folder
